In [1]:
# BINARY CLASSIFICATION 

import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
import wandb
import numpy as np
from sklearn.metrics import confusion_matrix
from tqdm import tqdm

# Initialize wandb
wandb.init(
    project="ablation-biggan",
    name="ResNet50, Epochs=20, BatchSize=32, LR=0.001, WeightDecay=1e-4",
    config={
        "epochs": 20,
        "batch_size": 32,
        "lr": 0.001,
        "optimizer": "Adam",
        "architecture": "resnet50",
        "img_size": 224,
        "split": [0.7, 0.15, 0.15],
        "weight_decay": 1e-4,
        "max_batches_per_epoch": 100,         # ✅ NEW
        "max_val_batches_per_epoch": 50,      # ✅ NEW
        "max_test_batches": 100               # ✅ NEW
    }
)
config = wandb.config

# Batch limits
max_batches = config.get("max_batches_per_epoch", None)
max_val_batches = config.get("max_val_batches_per_epoch", None)
max_test_batches = config.get("max_test_batches", None)

# Device
device = torch.device('cuda:1' if torch.cuda.is_available() else 'cpu')

# Transforms
transform = transforms.Compose([
    transforms.Resize((config.img_size, config.img_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# Dataset
dataset = datasets.ImageFolder(root='genimage/BigGAN/imagenet_ai_0419_biggan/train', transform=transform)
print("Class mapping:", dataset.class_to_idx)

# Train/Val/Test split
total_size = len(dataset)
train_size = int(config.split[0] * total_size)
val_size = int(config.split[1] * total_size)
test_size = total_size - train_size - val_size

train_ds, val_ds, test_ds = random_split(dataset, [train_size, val_size, test_size])
val_indices = val_ds.indices
test_indices = test_ds.indices

# DataLoaders
train_loader = DataLoader(train_ds, batch_size=config.batch_size, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=config.batch_size, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=config.batch_size, shuffle=False)

# Model
model = models.resnet50(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=config.lr, weight_decay=config.weight_decay)

# Store validation misclassified info
val_misclassified_indices = []
val_misclassified_paths = []

# Training loop
for epoch in range(config.epochs):
    model.train()
    train_loss = 0.0
    train_correct = 0
    total = 0

    train_loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{config.epochs} [Training]", leave=False)
    for batch_idx, (inputs, labels) in enumerate(train_loop):
        if max_batches is not None and batch_idx >= max_batches:
            break

        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        train_correct += (preds == labels).sum().item()
        total += labels.size(0)

        train_loop.set_postfix(loss=loss.item())

    train_acc = 100.0 * train_correct / total
    avg_train_loss = train_loss / (batch_idx + 1)

    # VALIDATION
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    val_loop = tqdm(val_loader, desc=f"Epoch {epoch+1}/{config.epochs} [Validation]", leave=False)
    with torch.no_grad():
        for val_batch_idx, (inputs, labels) in enumerate(val_loop):
            if max_val_batches is not None and val_batch_idx >= max_val_batches:
                break

            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

            val_loop.set_postfix(loss=loss.item())

            for i in range(len(labels)):
                global_index = val_indices[val_batch_idx * config.batch_size + i]
                if preds[i] != labels[i]:
                    val_misclassified_indices.append(global_index)
                    img_path, _ = dataset.samples[global_index]
                    val_misclassified_paths.append(img_path)

    val_acc = 100.0 * val_correct / val_total
    avg_val_loss = val_loss / (val_batch_idx + 1)

    # Log to wandb
    wandb.log({
        "epoch": epoch + 1,
        "train_loss": avg_train_loss,
        "train_accuracy": train_acc,
        "val_loss": avg_val_loss,
        "val_accuracy": val_acc
    })

    print(f"Epoch {epoch+1}: "
          f"Train Loss: {avg_train_loss:.4f}, Train Acc: {train_acc:.2f}%, "
          f"Val Loss: {avg_val_loss:.4f}, Val Acc: {val_acc:.2f}%")

# Save model
torch.save(model.state_dict(), "resnet_model.pth")
wandb.save("resnet_model.pth")

# Save validation misclassifications
with open("val_misclassified_indices.txt", "w") as f:
    for idx in val_misclassified_indices:
        f.write(f"{idx}\n")

with open("val_misclassified_paths.txt", "w") as f:
    for path in val_misclassified_paths:
        f.write(f"{path}\n")

print("Misclassified validation indices saved")
print("Misclassified validation paths saved")

# Final test evaluation
def evaluate_on_test(model, test_loader):
    model.eval()
    test_correct = 0
    test_total = 0
    test_loss = 0.0

    all_preds = []
    all_labels = []
    misclassified_indices = []
    misclassified_paths = []

    with torch.no_grad():
        for batch_idx, (inputs, labels) in enumerate(test_loader):
            if max_test_batches is not None and batch_idx >= max_test_batches:
                break

            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            test_loss += loss.item()

            _, preds = torch.max(outputs, 1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            for i in range(len(labels)):
                global_index = batch_idx * test_loader.batch_size + i
                if preds[i] != labels[i]:
                    misclassified_indices.append(global_index)
                    dataset_index = test_indices[global_index]
                    img_path, _ = dataset.samples[dataset_index]
                    misclassified_paths.append(img_path)

            test_correct += (preds == labels).sum().item()
            test_total += labels.size(0)

    test_acc = 100.0 * test_correct / test_total
    avg_test_loss = test_loss / (batch_idx + 1)

    print(f"\nFinal Test Accuracy: {test_acc:.2f}%, Test Loss: {avg_test_loss:.4f}")
    wandb.log({
        "test_accuracy": test_acc,
        "test_loss": avg_test_loss
    })

    wandb.log({
        "confusion_matrix": wandb.plot.confusion_matrix(
            probs=None,
            y_true=all_labels,
            preds=all_preds,
            class_names=list(dataset.class_to_idx.keys())
        )
    })

    with open("test_misclassified_indices.txt", "w") as f:
        for idx in misclassified_indices:
            f.write(f"{idx}\n")

    with open("test_misclassified_paths.txt", "w") as f:
        for path in misclassified_paths:
            f.write(f"{path}\n")

    print("Misclassified test indices saved")
    print("Misclassified test paths saved")

# Run test evaluation
evaluate_on_test(model, test_loader)


wandb: Currently logged in as: aarushishukla (aarushishukla-uc-santa-barbara) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Class mapping: {'ai': 0, 'nature': 1}


/home/aarushi/Detecting-Deepfakes/.venv/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/aarushi/Detecting-Deepfakes/.venv/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Epoch 1: Train Loss: 0.1875, Train Acc: 94.19%, Val Loss: 0.1002, Val Acc: 96.81%


Epoch 2: Train Loss: 0.1331, Train Acc: 95.97%, Val Loss: 0.1592, Val Acc: 94.69%


...

Epoch 17: Train Loss: 0.0835, Train Acc: 97.72%, Val Loss: 0.0775, Val Acc: 98.00%


Epoch 18: Train Loss: 0.0762, Train Acc: 97.91%, Val Loss: 0.1079, Val Acc: 96.75%


Epoch 19: Train Loss: 0.1257, Train Acc: 96.38%, Val Loss: 0.1072, Val Acc: 96.94%


Epoch 20: Train Loss: 0.0826, Train Acc: 97.62%, Val Loss: 0.0779, Val Acc: 98.00%
Misclassified validation indices saved
Misclassified validation paths saved


UnidentifiedImageError: cannot identify image file <_io.BufferedReader name='genimage/BigGAN/imagenet_ai_0419_biggan/train/ai/116_biggan_00107.png'>